Chatbot Reading Existing Memories

In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_core.messages import SystemMessage
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from dotenv import load_dotenv
import os
from typing import TypedDict,Annotated,List
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq


In [3]:
load_dotenv(dotenv_path=".env", override=True)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2" 
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3180.54it/s]


In [5]:
store=InMemoryStore(index={'embed':embedding_model,'dims':1536})

In [6]:
user_details=("user","u1")

In [7]:
store.put(user_details, "profile_1", {"data": "Name: Adnan"})
store.put(user_details, "profile_2", {"data": "Profession: AI Developer"})
store.put(user_details, "preference_1", {"data": "Prefers concise answers"})
store.put(user_details, "preference_2", {"data": "Likes examples in Python"})
store.put(user_details, "project_1", {"data": "Building AI Agents (Python-based project)"})

In [8]:
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, Nitish...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP  server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible. For example, instead of "In TypeScript apps..." 
say "Since your project is built with TypeScript..."

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [ ]:
class chatstate(TypedDict):
    messages:Annotated[]